In [1]:
import pandas as pd
import numpy as np
import re

df = pd.read_excel('oppo.xlsx')

df['MRP'] = df['MRP'].str.replace('₹', '').str.replace(' ', '').str.replace(',', '').astype(float)
df['Current Price'] = df['Current Price'].str.replace('₹', '').str.replace(',', '').astype(float)

def extract_discount(value):
    if pd.isna(value):
        return 0  # or return None
    return int(value.replace('% off', ''))
df['Discount_Number'] = df['Discount'].apply(extract_discount)

df.drop("Discount", axis=1, inplace=True)

brands = {
    'Apple': ['apple', 'iphone', 'ipad'],
    'motorola': ['motorola', 'moto'],
    'AI_plus': ['ai+', 'ai', 'aiplus', 'ai plus', 'a i'], 
    'IQOO': ['iqoo'],
    'Tecno': ['tecno'],
    'lava': ['lava'],
    'redmi': ['readmi', 'redmi'],  
    'realme': ['realme'],
    'POCO': ['poco'],
    'nothing': ['nothing'],
    'Infix': ['infix'],
    'Oppo': ['oppo'],
    'vivo': ['vivo'],
    'Google': ['google', 'pixel'],
    'OnePlus': ['oneplus', 'one plus'],
    'Samsung': ['samsung', 'galaxy'],
}

def get_brand(product_name):
    if pd.isna(product_name):
        return 'Unknown'
    
    product_name = str(product_name).lower()
    
    for brand, keywords in brands.items():
        for keyword in keywords:
            if keyword in product_name:
                return brand
    return 'Unknown'
df['Brand'] = df['Product Name'].apply(get_brand)

df['RAM'] = df['RAM'].str.extract(r'(\d+)').astype(int)


df['Storage'] = df['Storage'].str.extract(r'(\d+)').astype(int)


def extract_ratings_reviews(text):
    """
    Extract ratings and reviews from text
    Example: "2,46,375 Ratings & 9,265 Reviews"
    Returns: (ratings, reviews)
    """
    if pd.isna(text):
        return None, None
    
    text = str(text)
    
    # Extract ratings (numbers before 'Ratings')
    ratings_match = re.search(r'([\d,]+)\s*Ratings', text, re.IGNORECASE)
    ratings = ratings_match.group(1).replace(',', '') if ratings_match else None
    
    # Extract reviews (numbers before 'Reviews')
    reviews_match = re.search(r'([\d,]+)\s*Reviews', text, re.IGNORECASE)
    reviews = reviews_match.group(1).replace(',', '') if reviews_match else None
    
    return ratings, reviews

# Apply to dataframe
df[['Ratings', 'Reviews']] = df['Review text'].apply(
    lambda x: pd.Series(extract_ratings_reviews(x))
)

# Convert to numbers
df['Ratings'] = pd.to_numeric(df['Ratings'], errors='coerce')
df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce')


def extract_battery(text):
    if pd.isna(text):
        return 0
    match = re.search(r'(\d+)', str(text))
    return int(match.group()) if match else 0

df['Battery'] = df['Battery'].apply(extract_battery)

In [2]:
df.head()

,Product Name,Current Price,MRP,Rating,Review text,RAM,Storage,Display,Camera,Battery,Processor,Product URL,Image URL,Discount_Number,Brand,Ratings,Reviews
0,"OPPO K14 5G (Icy Blue, 256 GB)",21999.0,27999.0,4.4,"2,803 Ratings & 139 Reviews",6,256,17.14 cm (6.75 inch) HD+ Display,50MP Rear Camera | 8MP Front Camera,7000,Dimensity 6300 Processor,https://www.flipkart.com/oppo-k14-5g-icy-blue-...,https://rukminim2.flixcart.com/image/312/312/x...,21,Oppo,2803.0,139.0
1,"OPPO K14 5G (Prism White, 256 GB)",23999.0,29999.0,4.3,857 Ratings & 65 Reviews,8,256,17.14 cm (6.75 inch) HD+ Display,50MP Rear Camera | 8MP Front Camera,7000,Dimensity 6300 Processor,https://www.flipkart.com/oppo-k14-5g-prism-whi...,https://rukminim2.flixcart.com/image/312/312/x...,20,Oppo,857.0,65.0
2,"OPPO K14 5G (Prism White, 256 GB)",21999.0,27999.0,4.4,"2,803 Ratings & 139 Reviews",6,256,17.14 cm (6.75 inch) HD+ Display,50MP Rear Camera | 8MP Front Camera,7000,Dimensity 6300 Processor,https://www.flipkart.com/oppo-k14-5g-prism-whi...,https://rukminim2.flixcart.com/image/312/312/x...,21,Oppo,2803.0,139.0
3,"OPPO K14 5G (Prism Violet, 256 GB)",23999.0,29999.0,4.3,857 Ratings & 65 Reviews,8,256,17.14 cm (6.75 inch) HD+ Display,50MP Rear Camera | 8MP Front Camera,7000,Dimensity 6300 Processor,https://www.flipkart.com/oppo-k14-5g-prism-vio...,https://rukminim2.flixcart.com/image/312/312/x...,20,Oppo,857.0,65.0
4,"OPPO K14 5G (Prism Violet, 256 GB)",21999.0,27999.0,4.4,"2,803 Ratings & 139 Reviews",6,256,17.14 cm (6.75 inch) HD+ Display,50MP Rear Camera | 8MP Front Camera,7000,Dimensity 6300 Processor,https://www.flipkart.com/oppo-k14-5g-prism-vio...,https://rukminim2.flixcart.com/image/312/312/x...,21,Oppo,2803.0,139.0


In [3]:
df.isnull().sum()

Product Name         0
Current Price        3
MRP                103
Rating               1
Review text          1
RAM                  0
Storage              0
Display              0
Camera              29
Battery              0
Processor           12
Product URL          0
Image URL            0
Discount_Number      0
Brand                0
Ratings              1
Reviews              1
dtype: int64

In [4]:
df['MRP'] = np.where(
    (df['Discount_Number'] == 0) & (df['MRP'].isna() | (df['MRP'] == '')),
    df['Current Price'],
    df['MRP']
)

In [5]:
df.isnull().sum()

Product Name        0
Current Price       3
MRP                 3
Rating              1
Review text         1
RAM                 0
Storage             0
Display             0
Camera             29
Battery             0
Processor          12
Product URL         0
Image URL           0
Discount_Number     0
Brand               0
Ratings             1
Reviews             1
dtype: int64

In [6]:
df['Rating'] = df['Rating'].fillna(df['Rating'].median())
df['Ratings'] = df['Ratings'].fillna(df['Ratings'].median()).astype(int)
df['Reviews'] = df['Reviews'].fillna(df['Reviews'].median()).astype(int)

In [7]:
df['Current Price'] = df['Current Price'].fillna(df['Current Price'].median())
df['MRP'] = df['MRP'].fillna(df['Current Price'])

In [8]:
df.isnull().sum()

Product Name        0
Current Price       0
MRP                 0
Rating              0
Review text         1
RAM                 0
Storage             0
Display             0
Camera             29
Battery             0
Processor          12
Product URL         0
Image URL           0
Discount_Number     0
Brand               0
Ratings             0
Reviews             0
dtype: int64

In [9]:
def get_oppo_series(product_name):
    if pd.isna(product_name):
        return 'Unknown'
    
    # Find X Series (Flagship)
    if 'Find X' in product_name:
        if 'Find X8' in product_name:
            if 'Pro' in product_name:
                return 'Find X8 Pro'
            else:
                return 'Find X8'
        elif 'Find X7' in product_name:
            if 'Ultra' in product_name:
                return 'Find X7 Ultra'
            else:
                return 'Find X7'
        elif 'Find X6' in product_name:
            if 'Pro' in product_name:
                return 'Find X6 Pro'
            else:
                return 'Find X6'
        elif 'Find X5' in product_name:
            if 'Pro' in product_name:
                return 'Find X5 Pro'
            else:
                return 'Find X5'
        elif 'Find X3' in product_name:
            if 'Pro' in product_name:
                return 'Find X3 Pro'
            else:
                return 'Find X3'
        elif 'Find X2' in product_name:
            return 'Find X2'
        elif 'Find X' in product_name:
            return 'Find X Series'
    
    # Reno Series (Mid-range/Premium)
    elif 'Reno' in product_name:
        if 'Reno 13' in product_name:
            if 'Pro' in product_name:
                return 'Reno 13 Pro'
            else:
                return 'Reno 13'
        elif 'Reno 12' in product_name:
            if 'Pro' in product_name:
                return 'Reno 12 Pro'
            else:
                return 'Reno 12'
        elif 'Reno 11' in product_name:
            if 'Pro' in product_name:
                return 'Reno 11 Pro'
            else:
                return 'Reno 11'
        elif 'Reno 10' in product_name:
            if 'Pro+' in product_name:
                return 'Reno 10 Pro+'
            elif 'Pro' in product_name:
                return 'Reno 10 Pro'
            else:
                return 'Reno 10'
        elif 'Reno 9' in product_name:
            if 'Pro' in product_name:
                return 'Reno 9 Pro'
            else:
                return 'Reno 9'
        elif 'Reno 8' in product_name:
            if 'Pro' in product_name:
                return 'Reno 8 Pro'
            else:
                return 'Reno 8'
        elif 'Reno 7' in product_name:
            if 'Pro' in product_name:
                return 'Reno 7 Pro'
            else:
                return 'Reno 7'
        elif 'Reno 6' in product_name:
            return 'Reno 6'
        elif 'Reno' in product_name:
            return 'Reno Series'
    
    # A Series (Budget/Mid-range)
    elif 'A' in product_name and 'A' != product_name:
        if 'A79' in product_name:
            return 'A79'
        elif 'A78' in product_name:
            return 'A78'
        elif 'A77' in product_name:
            return 'A77'
        elif 'A76' in product_name:
            return 'A76'
        elif 'A75' in product_name:
            return 'A75'
        elif 'A74' in product_name:
            return 'A74'
        elif 'A60' in product_name:
            return 'A60'
        elif 'A57' in product_name:
            return 'A57'
        elif 'A55' in product_name:
            return 'A55'
        elif 'A54' in product_name:
            return 'A54'
        elif 'A53' in product_name:
            return 'A53'
        elif 'A52' in product_name:
            return 'A52'
        elif 'A' in product_name:
            return 'A Series'
    
    # F Series (Selfie-focused)
    elif 'F' in product_name:
        return 'F Series'
    
    # K Series
    elif 'K' in product_name:
        return 'K Series'
    
    else:
        return 'Other'

df['Oppo_Series'] = df['Product Name'].apply(get_oppo_series)

def get_oppo_camera(product_name):
    if pd.isna(product_name):
        return 'Unknown Camera'
    
    # Find X Series
    if 'Find X8 Pro' in product_name:
        return '50MP + 50MP + 50MP'
    elif 'Find X8' in product_name:
        return '50MP + 50MP + 50MP'
    elif 'Find X7 Ultra' in product_name:
        return '50MP + 50MP + 50MP + 50MP'
    elif 'Find X7' in product_name:
        return '50MP + 50MP + 50MP'
    elif 'Find X6 Pro' in product_name:
        return '50MP + 50MP + 50MP'
    elif 'Find X6' in product_name:
        return '50MP + 50MP + 32MP'
    elif 'Find X5 Pro' in product_name:
        return '50MP + 50MP + 13MP'
    elif 'Find X5' in product_name:
        return '50MP + 50MP + 13MP'
    elif 'Find X3 Pro' in product_name:
        return '50MP + 50MP + 13MP'
    elif 'Find X3' in product_name:
        return '50MP + 50MP + 13MP'
    elif 'Find X2' in product_name:
        return '48MP + 13MP + 12MP'
    
    # Reno Series
    elif 'Reno 13 Pro' in product_name:
        return '50MP + 8MP + 50MP'
    elif 'Reno 13' in product_name:
        return '50MP + 8MP + 2MP'
    elif 'Reno 12 Pro' in product_name:
        return '50MP + 8MP + 50MP'
    elif 'Reno 12' in product_name:
        return '50MP + 8MP + 2MP'
    elif 'Reno 11 Pro' in product_name:
        return '50MP + 32MP + 8MP'
    elif 'Reno 11' in product_name:
        return '64MP + 8MP + 2MP'
    elif 'Reno 10 Pro+' in product_name:
        return '50MP + 8MP + 64MP'
    elif 'Reno 10 Pro' in product_name:
        return '50MP + 8MP + 32MP'
    elif 'Reno 10' in product_name:
        return '64MP + 8MP + 2MP'
    elif 'Reno 9 Pro' in product_name:
        return '50MP + 8MP + 2MP'
    elif 'Reno 9' in product_name:
        return '64MP + 2MP'
    elif 'Reno 8 Pro' in product_name:
        return '50MP + 8MP + 2MP'
    elif 'Reno 8' in product_name:
        return '64MP + 2MP'
    elif 'Reno 7 Pro' in product_name:
        return '50MP + 8MP + 2MP'
    elif 'Reno 7' in product_name:
        return '64MP + 2MP'
    elif 'Reno 6' in product_name:
        return '64MP + 8MP + 2MP'
    
    # A Series
    elif 'A79' in product_name:
        return '50MP + 2MP'
    elif 'A78' in product_name:
        return '50MP + 2MP'
    elif 'A77' in product_name:
        return '50MP + 2MP'
    elif 'A76' in product_name:
        return '13MP + 2MP'
    elif 'A75' in product_name:
        return '50MP + 2MP'
    elif 'A74' in product_name:
        return '50MP + 2MP'
    elif 'A60' in product_name:
        return '50MP + 2MP'
    elif 'A57' in product_name:
        return '13MP + 2MP'
    elif 'A55' in product_name:
        return '50MP + 2MP'
    elif 'A54' in product_name:
        return '50MP + 2MP'
    elif 'A53' in product_name:
        return '50MP + 2MP'
    elif 'A52' in product_name:
        return '50MP + 2MP'
    elif 'A' in product_name:
        return '50MP + 2MP'
    
    # F Series
    elif 'F' in product_name:
        return '64MP + 2MP'
    
    # K Series
    elif 'K' in product_name:
        return '50MP + 2MP'
    
    else:
        return 'Unknown Camera'

df.loc[df['Camera'].isnull(), 'Camera'] = df.loc[df['Camera'].isnull(), 'Product Name'].apply(get_oppo_camera)


def get_oppo_processor(product_name):
    if pd.isna(product_name):
        return 'Unknown Processor'
    
    # Find X Series
    if 'Find X8 Pro' in product_name or 'Find X8' in product_name:
        return 'MediaTek Dimensity 9400'
    elif 'Find X7 Ultra' in product_name or 'Find X7' in product_name:
        return 'MediaTek Dimensity 9300'
    elif 'Find X6 Pro' in product_name:
        return 'Snapdragon 8 Gen 2'
    elif 'Find X6' in product_name:
        return 'MediaTek Dimensity 9200'
    elif 'Find X5 Pro' in product_name:
        return 'Snapdragon 8 Gen 1'
    elif 'Find X5' in product_name:
        return 'Snapdragon 888'
    elif 'Find X3 Pro' in product_name:
        return 'Snapdragon 888'
    elif 'Find X3' in product_name:
        return 'Snapdragon 870'
    elif 'Find X2' in product_name:
        return 'Snapdragon 865'
    
    # Reno Series
    elif 'Reno 13 Pro' in product_name:
        return 'MediaTek Dimensity 8350'
    elif 'Reno 13' in product_name:
        return 'MediaTek Dimensity 8350'
    elif 'Reno 12 Pro' in product_name:
        return 'MediaTek Dimensity 7300'
    elif 'Reno 12' in product_name:
        return 'MediaTek Dimensity 7300'
    elif 'Reno 11 Pro' in product_name:
        return 'Snapdragon 8+ Gen 1'
    elif 'Reno 11' in product_name:
        return 'MediaTek Dimensity 7050'
    elif 'Reno 10 Pro+' in product_name:
        return 'Snapdragon 8 Gen 1'
    elif 'Reno 10 Pro' in product_name:
        return 'MediaTek Dimensity 8200'
    elif 'Reno 10' in product_name:
        return 'Snapdragon 778G'
    elif 'Reno 9 Pro' in product_name:
        return 'MediaTek Dimensity 8100'
    elif 'Reno 9' in product_name:
        return 'Snapdragon 695'
    elif 'Reno 8 Pro' in product_name:
        return 'Snapdragon 7 Gen 1'
    elif 'Reno 8' in product_name:
        return 'MediaTek Dimensity 1300'
    elif 'Reno 7 Pro' in product_name:
        return 'MediaTek Dimensity 1200'
    elif 'Reno 7' in product_name:
        return 'Snapdragon 680'
    elif 'Reno 6' in product_name:
        return 'MediaTek Dimensity 900'
    
    # A Series
    elif 'A79' in product_name:
        return 'MediaTek Dimensity 6020'
    elif 'A78' in product_name:
        return 'Snapdragon 680'
    elif 'A77' in product_name:
        return 'Snapdragon 680'
    elif 'A76' in product_name:
        return 'Snapdragon 662'
    elif 'A75' in product_name:
        return 'MediaTek Dimensity 700'
    elif 'A74' in product_name:
        return 'Snapdragon 695'
    elif 'A60' in product_name:
        return 'MediaTek Helio G80'
    elif 'A57' in product_name:
        return 'MediaTek Helio G35'
    elif 'A55' in product_name:
        return 'Snapdragon 680'
    elif 'A54' in product_name:
        return 'Snapdragon 680'
    elif 'A53' in product_name:
        return 'Snapdragon 680'
    elif 'A52' in product_name:
        return 'Snapdragon 665'
    elif 'A' in product_name:
        return 'Snapdragon 680'
    
    # F Series
    elif 'F' in product_name:
        return 'Snapdragon 680'
    
    # K Series
    elif 'K' in product_name:
        return 'Snapdragon 695'
    
    else:
        return 'Unknown Processor'

df.loc[df['Processor'].isnull(), 'Processor'] = df.loc[df['Processor'].isnull(), 'Product Name'].apply(get_oppo_processor)

In [10]:
df.isnull().sum()

Product Name       0
Current Price      0
MRP                0
Rating             0
Review text        1
RAM                0
Storage            0
Display            0
Camera             0
Battery            0
Processor          0
Product URL        0
Image URL          0
Discount_Number    0
Brand              0
Ratings            0
Reviews            0
Oppo_Series        0
dtype: int64

In [11]:
df.drop("Oppo_Series", axis=1, inplace=True)

In [12]:
df.to_excel('Cleaned_oppo.xlsx', index=False)